FINAL GOLD MODEL
1. Fact Table → fact_sales
Granularity: 1 row = 1 order line item

Columns:
order_number        (FK)/
customer_id         (FK)
product_id / item_id
order_date_key      (FK)

qty
unit_price (if available)
discount
promo_quantity

total_amount (derived)
Derived:
total_amount = qty * unit_price * (1 - discount)

In [0]:
# ============================================================
# 0. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")
# ============================================================
# 1. IMPORTS
# ============================================================
import json
import pyspark.sql.functions as f
from common_utils.generic_functions import  casting
from common_utils.logging import get_logger

# ============================================================
# 2. LOGGER
# ============================================================
logger = get_logger("fact_sales.ipynb")
logger.info("Session for fact_sales started")


# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info(
    "Reading Gold configuration from %s",config_path
)
with open(config_path,"r") as file:
    config = json.load(file)

logger.info("Gold configuration loaded successfully")


# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================
sales_order_config = config["sales_order"]
sales_config = config["sales"]
promotions_config = config["promotions"]
target_config= config["target"]

sales_order_source_name = sales_order_config["source_name"]
sales_order_source_table = sales_order_config["source_table"]
logger.info(
    "[%s] Source table: %s",
    sales_order_source_name,
    sales_order_source_table
)

sales_source_name = sales_config["source_name"]
sales_source_table = sales_config["source_table"]
logger.info(
    "[%s] Source table: %s",
    sales_source_name,
    sales_source_table
)

promotions_source_name = promotions_config["source_name"]
promotions_source_table = promotions_config["source_table"]
logger.info(
    "[%s] Source table: %s",
    promotions_source_name,
    promotions_source_table
)

target_table = config["target"]["target_table"]
source_name  = config["source"]["source_name"]
logger.info(
    "[%s] Reading",
    source_name
)



# ============================================================
# 5. READ SOURCE TABLE
# ============================================================
# sales_order
logger.info(
    "[%s] Reading customers from %s",
    sales_order_source_name,
    sales_order_source_table
)

sales_order_df = spark.read.table(sales_order_source_table)

logger.info(
    "[%s] Sales_order table loaded successfully",
    sales_order_source_name
)

#sales

logger.info(
    "[%s] Reading sales from %s",
    sales_source_name,
    sales_source_table
)

sales_df = spark.read.table(sales_source_table)

logger.info(
    "[%s] Sales table loaded successfully",
    sales_source_name
)

#promotion table

logger.info(
    "[%s] Reading promotions details from %s",
    promotions_source_name,
    promotions_source_table
)

promotions_df = spark.read.table(promotions_source_table)

logger.info(
    "[%s] Promotion table loaded successfully",
    promotions_source_name
)

#orders_products table

orders_products_df = spark.read.table("retail_data_platform.silver.orders_products")

logger.info("orders_products table loaded successfully")



# ============================================================
# 6. TRANSFORMATIONS
# ============================================================
# now herem we going to work on transformation we are going to work on each table one by one and will move one after getting the required info to the next step


logger.info("The transformations for fact table has been started")

# ============================================================
# 6.1 promotions
# ============================================================
def transform_promotion(df,config):
    df =  df.select("order_number","promo_product_id", "discount","promo_quantity")

    df = casting(df, config["promotion_casting"])

    #now im going to remove the duplicates using the groupby and the keys for that gonna be order_number and promo_product_id , I dont want the two values in the fact table and long with that I will use the f.max function to get the max discount and max promo_quantity

    df = df.groupby("order_number","promo_product_id").agg(f.max("discount").alias("discount"), f.max("promo_quantity").alias("promo_quantity"))


    logger.info("promotion prepaired successfully")

    return df

# ============================================================
# 7 Transform fact sales
# ============================================================

def transform_fact_sales(promotion_df , sales_order_df , sales_df, config):
    logger.info("[%s] Starting fact_sales transformation",source_name)


    # ========================================================
    # 7.1 PREPARE ORDERS
    # ========================================================
    logger.info(
        "[%s] Preparing sales_orders",
        source_name
    )
    orders = (sales_order_df.select("customer_id","order_number","order_timestamp").withColumn("order_date_key",f.date_format(f.to_timestamp(f.col("order_timestamp")),config["transform"]["order_date_key_format"]).cast("int")).select("order_number","customer_id","order_date_key"))

    # ========================================================
    # 7.2 PREPARE order product
    # ========================================================
    logger.info("[%s] prepairing order_products",source_name)
    product = (
        sales_df.select("order_number","customer_id",f.col("qty").alias("qty"),f.col("id").alias("product_id"),f.col("price").alias("unit_price"))
    )

    # ========================================================
    # 7.3 PREPARE PROMOTIONS
    # ========================================================

    promotions = transform_promotion(promotion_df,config)


    # ========================================================
    # 7.4 JOIN ORDERS + PRODUCTS
    # ========================================================
    logger.info("[%s] Joining sales_orders with order_products",source_name)

    fact = (product.alias("p").join(orders.alias("o"),f.col("p.order_number")==f.col("o.order_number"),how="left"))

    # ========================================================
    # 7.5 JOIN PROMOTIONS
    # ========================================================
    logger.info("joining promotions for fact_sales")

    fact = (fact.join(promotions.alias("pro"),(f.col("p.product_id") == f.col("pro.promo_product_id"))&(f.col("p.order_number")==f.col("pro.order_number")),how="left"))

    
    # ========================================================
    # 7.6 SELECT FACT COLUMNS
    # ========================================================

    logger.info("[%s] Selecting fact_sales columns",source_name)

    fact = fact.select(
        # ----------------------------------------------------
        # Keys
        # ----------------------------------------------------
        f.col("p.order_number").alias("order_number"),
        f.col("o.customer_id").alias("customer_id"),
        f.col("p.product_id").alias("product_id"),
        f.col("o.order_date_key").alias("order_date_key"),
        # ----------------------------------------------------
        # Measures
        # ----------------------------------------------------
        f.col("p.qty").alias("qty"),
        f.col("p.unit_price").alias("unit_price"),

        # ----------------------------------------------------
        # Promotion information
        # ----------------------------------------------------
        f.coalesce(f.col("pro.discount"),f.lit(config["transform"]["default_discount"])).alias("discount"),
        f.coalesce(f.col("pro.promo_quantity"),f.lit(config["transform"]["default_promo_quantity"])).alias("promo_quantity")
    )
        
    logger.info(
        "[%s] Calculating total_amount",
        source_name
    )
    
    # Creating Total Amount columns 
        
    fact = fact.withColumn("total_amount",f.round(f.col("qty")*f.col("unit_price")*(f.lit(1)-f.col("discount")),2))
        


    # ========================================================
    # 7.7 CAST FINAL COLUMNS
    # ========================================================

    logger.info("[%s] Applying final data types",source_name)

    fact = casting(fact,config.get("cast_columns", {}))

    # ========================================================
    # 7.8 SELECT FINAL COLUMNS
    # ========================================================

    fact = fact.select(
        *config["select_columns"]
    )

    # ========================================================
    # 7.9 REMOVE DUPLICATES
    # ========================================================

    logger.info(
        "[%s] Removing duplicate fact records",
        source_name
    )

    fact = fact.dropDuplicates(
        [
            "order_number",
            "product_id"
        ]
    )


    logger.info(
        "[%s] fact_sales transformation completed",
        source_name
    )


    return fact

# ============================================================
# 8. CREATE FACT SALES
# ============================================================

logger.info("[%s] Starting Gold fact_sales creation",source_name)

fact_sales = transform_fact_sales(
    promotion_df=promotions_df,
    sales_order_df=sales_order_df,
    sales_df=orders_products_df,
    config=config
)

# ============================================================
# 9. VALIDATE FACT DATA
# ============================================================

logger.info("[%s] Validating fact_sales",source_name)
record_count = fact_sales.count()
logger.info("[%s] fact_sales record count: %s",source_name,record_count)


if record_count == 0:
    raise ValueError(
        "fact_sales contains zero records"
    )


# Check for duplicate grain

duplicate_count = (
    fact_sales
    .groupBy(
        "order_number",
        "product_id"
    )
    .count()
    .filter(
        f.col("count") > 1
    )
    .count()
)

logger.info("[%s] Duplicate fact grain count: %s",source_name,duplicate_count)


if duplicate_count > 0:
    raise ValueError(
        "Duplicate order_number + product_id "
        "records detected in fact_sales"
    )

# ============================================================
# 10. DISPLAY FACT
# ============================================================

fact_sales.show()

# ============================================================
# 11. CREATE GOLD SCHEMA
# ============================================================

logger.info("[%s] Creating Gold schema if required",source_name)

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    retail_data_platform.gold
""")

# ============================================================
# 12. WRITE GOLD TABLE
# ============================================================

logger.info("[%s] Writing fact_sales to %s",source_name,target_table)


(
    fact_sales
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(target_table)
)




logger.info("[%s] Gold table %s written successfully",source_name,target_table)




# ============================================================
# 13. FINAL VALIDATION
# ============================================================

logger.info("[%s] Running final Gold table validation",source_name)
final_count = (spark.read.table(target_table).count())
logger.info("[%s] Final Gold record count: %s",source_name,final_count)
logger.info("[%s] Gold fact_sales pipeline completed successfully",source_name)
